In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from models.LogisticRegression import LogisticRegressionPytorch

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
dataloader = DataLoader(
    Elastic(timeout=5)
)

In [15]:
X = dataloader.lex_df.drop('type', axis=1, inplace=False)
y = dataloader.lex_df['type']

X_train, X_test, y_train, y_test = dataloader.train_test_split(X, y, train_split=0.8, random_state=15)

In [16]:
X_train

,len_url,len_component,count_digits_component,count_letters_component,ratio_digits_component_url,ratio_letters_component_url,count_dots_url,count_percent_url,count_hash_url,count_ats_url,count_embed_url,use_https,no_of_directories,contains_ip_address,character_continuity_rate_url,shannon_entropy_url
8631,35,1,0,0,0.000000,0.000000,2,0,0,0,1,1,1,0,0.114286,4.150293
7983,27,1,0,0,0.000000,0.000000,2,0,0,0,1,1,1,0,0.074074,3.810081
7082,69,19,0,17,0.000000,0.246377,3,0,0,0,1,0,1,0,0.072464,4.708476
3265,41,1,0,0,0.000000,0.000000,1,0,0,0,1,1,1,0,0.048780,4.082804
3001,25,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.080000,3.813661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5483,6,6,2,3,0.333333,0.500000,1,0,0,0,0,0,0,0,0.000000,2.584963
6892,25,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.080000,3.833270
4832,35,11,0,9,0.000000,0.257143,2,0,0,0,1,0,2,0,0.057143,4.214997
8076,20,20,0,19,0.000000,0.950000,1,0,0,0,0,0,0,0,0.000000,3.884184


In [17]:
X_test

,len_url,len_component,count_digits_component,count_letters_component,ratio_digits_component_url,ratio_letters_component_url,count_dots_url,count_percent_url,count_hash_url,count_ats_url,count_embed_url,use_https,no_of_directories,contains_ip_address,character_continuity_rate_url,shannon_entropy_url
7866,25,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.080000,3.863465
1572,41,1,0,0,0.000000,0.000000,3,0,0,0,1,0,1,0,0.073171,4.153062
4118,25,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.080000,3.733661
1992,95,13,4,7,0.042105,0.073684,4,0,0,0,1,1,1,0,0.063158,4.886557
1212,20,1,0,0,0.000000,0.000000,2,0,0,0,1,1,1,0,0.100000,3.746439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
966,25,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.080000,3.653661
4846,21,1,0,0,0.000000,0.000000,2,0,0,0,1,0,1,0,0.142857,3.558519
5519,14,14,0,13,0.000000,0.928571,1,0,0,0,0,0,0,0,0.000000,3.378783
4568,18,18,0,17,0.000000,0.944444,1,0,0,0,0,0,0,0,0.111111,3.239098


## ScikitLearn Model

In [11]:
onehot_hyperparams = {
    'categories': 'auto', 
    'drop': None, 
    'dtype': np.float64, 
    'handle_unknown': 'error', 
    'min_frequency': None, 
    'max_categories': None, 
    'feature_name_combiner': 'concat'
}

lr_hyperparams = {
    'penalty': 'l2',
    'dual': False,
    'tol': 1e-4,
    'C': 1,
    'fit_intercept': True,
    'intercept_scaling': 1,
    'class_weight': None,
    'random_state': None,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'multi_class': 'auto',
    'verbose': 0,
    'warm_start': False,
    'n_jobs': None,
    'l1_ratio': None
}

In [18]:
pipe = Pipeline(
    [
        ('stdscaler', StandardScaler()), 
        #('onehotencoder', OneHotEncoder(**onehot_hyperparams))
        ('lrmodel', LogisticRegression(**lr_hyperparams))
    ]
)

pipe.fit(X_train, y_train).score(X_test, y_test)

1.0

## Pytorch Model

In [19]:
input_size = X_train.shape[1]
model = LogisticRegressionPytorch(input_size)

_hyperparams = {
    'lr': 0.01, 
    'momentum': 0,
    'dampening': 0,
    'weight_decay': 0,
    'nesterov': False
}

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), **_hyperparams)

num_epochs = 1000

In [22]:
X_train_tensor = torch.from_numpy(X_train.to_numpy().astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.to_numpy().astype(np.float32))

X_test_tensor = torch.from_numpy(X_test.to_numpy().astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.to_numpy().astype(np.float32))

In [23]:
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor.view(-1, 1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 0.0919
Epoch [200/1000], Loss: 0.0563
Epoch [300/1000], Loss: 0.0402
Epoch [400/1000], Loss: 0.0311
Epoch [500/1000], Loss: 0.0253
Epoch [600/1000], Loss: 0.0213
Epoch [700/1000], Loss: 0.0183
Epoch [800/1000], Loss: 0.0161
Epoch [900/1000], Loss: 0.0143
Epoch [1000/1000], Loss: 0.0129


In [24]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predictions = (predictions > 0.5).float()

accuracy = (predictions == y_test_tensor.view(-1, 1)).sum().item() / len(y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 99.96%


In [ ]:
# Export the model
torch.onnx.export(model,               # model being run
                  X_train_tensor,                         # model input (or a tuple for multiple inputs)
                  "model.onnx",   # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  #opset_version=10,          # the ONNX version to export the model to
                  #do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['input'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}})